# PySpark Day 2 — DataFrame Transformations
**Business dataset:** Retail orders  
**Assignment domain:** Banking

Goal: practice narrow/wide transformations and build business-ready retail outputs.

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder.appName("RetailPySparkTraining").getOrCreate()

retail_data = [
    (1001,"Laptop","Electronics",65000.0,2,0.10,"Bengaluru","2026-09-01"),
    (1002,"Phone","Electronics",30000.0,3,0.05,"Hyderabad","2026-09-02"),
    (1003,"Shoes","Fashion",4500.0,5,0.20,"Chennai","2026-09-02"),
    (1004,"Watch","Fashion",8000.0,2,0.15,"Bengaluru","2026-09-03"),
    (1005,"TV","Electronics",55000.0,1,0.10,"Pune","2026-09-03"),
    (1006,"Chair","Furniture",7000.0,4,0.05,"Hyderabad","2026-09-04"),
    (1007,"Table","Furniture",12000.0,2,0.10,"Chennai","2026-09-04"),
    (1008,"Headphones","Electronics",3500.0,6,0.00,"Bengaluru","2026-09-05"),
    (1009,"Bag","Fashion",2500.0,8,0.10,"Pune","2026-09-05"),
    (1010,"Sofa","Furniture",40000.0,1,0.12,"Bengaluru","2026-09-06")
]
cols=["order_id","product","category","price","quantity","discount","city","order_date"]
df=spark.createDataFrame(retail_data,cols)
df.show(truncate=False)

## 1. `select()` — choose required columns

In [ ]:
df.select("order_id","product","price").show()

## 2. `filter()` / `where()` — filter business records

In [ ]:
df.filter(F.col("price") > 10000).show()
df.where(F.col("category") == "Electronics").show()

## 3. `withColumn()` — derive gross and net sales

In [ ]:
sales_df=(df
 .withColumn("gross_sales", F.col("price")*F.col("quantity"))
 .withColumn("net_sales", F.col("price")*F.col("quantity")*(1-F.col("discount"))))
sales_df.show()

## 4. `withColumnRenamed()`

In [ ]:
df.withColumnRenamed("city","customer_city").show(3)

## 5. `drop()`

In [ ]:
df.drop("discount").show(3)

## 6. `distinct()` and `dropDuplicates()`

In [ ]:
df.select("category").distinct().show()
df.dropDuplicates(["order_id"]).show()

## 7. `orderBy()` / `sort()`

In [ ]:
df.orderBy(F.col("price").desc()).show()

## 8. `groupBy()` + aggregations

In [ ]:
df.groupBy("category").agg(
    F.count("order_id").alias("orders"),
    F.sum("quantity").alias("units"),
    F.avg("price").alias("avg_price"),
    F.max("price").alias("max_price")
).show()

## 9. Conditional transformation with `when/otherwise`

In [ ]:
df.withColumn("price_band",
    F.when(F.col("price")>=50000,"Premium")
     .when(F.col("price")>=10000,"Mid")
     .otherwise("Budget")
).show()

## 10. String transformations

In [ ]:
df.select(
    F.upper("product").alias("product_upper"),
    F.lower("category").alias("category_lower"),
    F.length("product").alias("name_length")
).show()

## 11. Date transformations

In [ ]:
date_df=df.withColumn("order_date",F.to_date("order_date"))
date_df.select("order_id","order_date",F.year("order_date").alias("year"),F.month("order_date").alias("month"),F.dayofmonth("order_date").alias("day")).show()

## 12. Null handling transformations

In [ ]:
df.fillna({"discount":0.0}).dropna(subset=["order_id","product"]).show()

## 13. `unionByName()`

In [ ]:
extra=spark.createDataFrame([(1011,"Keyboard","Electronics",2500.0,3,0.05,"Chennai","2026-09-07")],df.columns)
df.unionByName(extra).show()

## 14. Join transformation

In [ ]:
category_data=[("Electronics",0.18),("Fashion",0.12),("Furniture",0.15)]
category_df=spark.createDataFrame(category_data,["category","tax_rate"])
df.join(category_df,"category","left").show()

## 15. Repartition and coalesce

In [ ]:
print("Original partitions:",df.rdd.getNumPartitions())
repart_df=df.repartition(4,"category")
print("After repartition:",repart_df.rdd.getNumPartitions())
coal_df=repart_df.coalesce(2)
print("After coalesce:",coal_df.rdd.getNumPartitions())

## 16. Retail transformation pipeline

In [ ]:
final_df=(df
 .filter(F.col("quantity")>0)
 .withColumn("gross_sales",F.col("price")*F.col("quantity"))
 .withColumn("net_sales",F.round(F.col("gross_sales")*(1-F.col("discount")),2))
 .withColumn("order_date",F.to_date("order_date"))
 .select("order_id","product","category","city","order_date","quantity","gross_sales","net_sales")
 .orderBy(F.col("net_sales").desc()))
final_df.show(truncate=False)

## Day 2 Banking Assignment
Create/use a banking transactions DataFrame with: `transaction_id, customer_id, account_type, transaction_type, amount, city, transaction_date`.

Tasks:
1. Select transaction ID, customer ID and amount.
2. Filter transactions above ₹50,000.
3. Filter only `DEBIT` transactions.
4. Create `transaction_fee` as 1% of amount.
5. Create `net_amount` after fee.
6. Rename `city` to `branch_city`.
7. Drop an unnecessary column.
8. Find distinct account types.
9. Remove duplicate transaction IDs.
10. Sort transactions by amount descending.
11. Group by account type and calculate count, total amount, average amount, min and max.
12. Create `risk_level`: HIGH >= ₹100,000, MEDIUM >= ₹50,000, otherwise LOW.
13. Convert transaction date to DateType and extract year/month/day.
14. Handle null amount/city values appropriately.
15. Create another DataFrame and perform `unionByName`.
16. Create a customer master DataFrame and join it with transactions.
17. Repartition by account type and explain why.
18. Build a final banking pipeline producing clean, enriched transactions.

**Challenge:** Classify the transformations you used as narrow or wide and identify where a shuffle is likely to occur.

## Banking Assignment — Solutions
### Create the banking transactions DataFrame
The data is designed to exercise every task:
- a mix of `DEBIT` and `CREDIT` transactions
- amounts in all risk bands (HIGH >= ₹100,000, MEDIUM >= ₹50,000, LOW)
- one **duplicate** transaction (`5002`) for Task 9
- one **null amount** (`5009`) and one **null city** (`5006`) for Task 14

An explicit schema is used so the null values get the correct data types.

In [ ]:
bank_schema = StructType([
    StructField("transaction_id", LongType(), False),
    StructField("customer_id", StringType(), True),
    StructField("account_type", StringType(), True),
    StructField("transaction_type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("city", StringType(), True),
    StructField("transaction_date", StringType(), True)
])

banking_data = [
    (5001,"C101","Savings","CREDIT",25000.0,"Bengaluru","2026-09-01"),
    (5002,"C102","Current","DEBIT",120000.0,"Hyderabad","2026-09-01"),
    (5003,"C103","Salary","CREDIT",85000.0,"Chennai","2026-09-02"),
    (5004,"C101","Savings","DEBIT",5000.0,"Bengaluru","2026-09-02"),
    (5005,"C104","Current","DEBIT",65000.0,"Pune","2026-09-03"),
    (5006,"C105","Savings","DEBIT",7000.0,None,"2026-09-03"),
    (5007,"C102","Current","CREDIT",150000.0,"Hyderabad","2026-09-04"),
    (5008,"C106","Salary","DEBIT",15000.0,"Chennai","2026-09-04"),
    (5009,"C103","Salary","DEBIT",None,"Chennai","2026-09-05"),
    (5010,"C107","Savings","CREDIT",10000.0,"Pune","2026-09-05"),
    (5011,"C104","Current","DEBIT",52000.0,"Pune","2026-09-06"),
    (5012,"C108","Savings","DEBIT",1500.0,"Mumbai","2026-09-06"),
    (5013,"C105","Savings","CREDIT",8000.0,"Mumbai","2026-09-07"),
    (5014,"C109","Salary","CREDIT",105000.0,"Bengaluru","2026-09-07"),
    (5015,"C110","Current","DEBIT",30000.0,"Hyderabad","2026-09-08"),
    (5002,"C102","Current","DEBIT",120000.0,"Hyderabad","2026-09-01")   # duplicate
]
bank_df=spark.createDataFrame(banking_data,bank_schema)
bank_df.show(truncate=False)

### 1. Select transaction ID, customer ID and amount

In [ ]:
bank_df.select("transaction_id","customer_id","amount").show()

### 2. Filter transactions above ₹50,000

In [ ]:
bank_df.filter(F.col("amount") > 50000).show()

### 3. Filter only `DEBIT` transactions

In [ ]:
bank_df.where(F.col("transaction_type") == "DEBIT").show()

### 4. Create `transaction_fee` as 1% of amount

In [ ]:
fee_df=bank_df.withColumn("transaction_fee", F.round(F.col("amount")*0.01,2))
fee_df.show()

### 5. Create `net_amount` after fee

In [ ]:
net_df=fee_df.withColumn("net_amount", F.round(F.col("amount")-F.col("transaction_fee"),2))
net_df.select("transaction_id","amount","transaction_fee","net_amount").show()

### 6. Rename `city` to `branch_city`

In [ ]:
bank_df.withColumnRenamed("city","branch_city").show(3)

### 7. Drop an unnecessary column
Once `net_amount` is calculated, the intermediate `transaction_fee` column is not needed in a customer statement, so drop it.

In [ ]:
net_df.drop("transaction_fee").show(3)

### 8. Find distinct account types

In [ ]:
bank_df.select("account_type").distinct().show()

### 9. Remove duplicate transaction IDs

In [ ]:
dedup_df=bank_df.dropDuplicates(["transaction_id"])
print("Before:",bank_df.count(),"| After:",dedup_df.count())
dedup_df.orderBy("transaction_id").show()

### 10. Sort transactions by amount descending

In [ ]:
dedup_df.orderBy(F.col("amount").desc()).show()

### 11. Group by account type — count, total, average, min, max
Uses the de-duplicated data so transaction `5002` isn't counted twice. Aggregate functions ignore the null amount.

In [ ]:
dedup_df.groupBy("account_type").agg(
    F.count("transaction_id").alias("txn_count"),
    F.sum("amount").alias("total_amount"),
    F.round(F.avg("amount"),2).alias("avg_amount"),
    F.min("amount").alias("min_amount"),
    F.max("amount").alias("max_amount")
).show()

### 12. Create `risk_level`

In [ ]:
dedup_df.withColumn("risk_level",
    F.when(F.col("amount")>=100000,"HIGH")
     .when(F.col("amount")>=50000,"MEDIUM")
     .otherwise("LOW")
).select("transaction_id","amount","risk_level").show()

### 13. Convert transaction date to DateType and extract year/month/day

In [ ]:
date_df=dedup_df.withColumn("transaction_date",F.to_date("transaction_date"))
date_df.printSchema()
date_df.select("transaction_id","transaction_date",
               F.year("transaction_date").alias("year"),
               F.month("transaction_date").alias("month"),
               F.dayofmonth("transaction_date").alias("day")).show()

### 14. Handle null amount/city values
- **amount:** a transaction with no amount can't be trusted, and filling in `0` would give wrong totals. **Drop** the row (in a real bank it would go to a reject/audit table).
- **city:** the transaction is still valid, so **fill** the missing branch with `"Unknown"`.

In [ ]:
dedup_df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in ["amount","city"]]).show()

rejected_df=dedup_df.filter(F.col("amount").isNull())
clean_df=dedup_df.dropna(subset=["amount"]).fillna({"city":"Unknown"})

print("Rejected rows:")
rejected_df.show()
print("Clean rows:",clean_df.count())
clean_df.show()

### 15. Create another DataFrame and perform `unionByName`
The new DataFrame's columns are in a **different order**. `unionByName` matches columns by name, whereas `union` matches by position and would put values in the wrong columns.

In [ ]:
new_txn=spark.createDataFrame(
    [("C111",5016,"Savings","CREDIT",45000.0,"Kolkata","2026-09-09"),
     ("C101",5017,"Savings","DEBIT",110000.0,"Bengaluru","2026-09-09")],
    ["customer_id","transaction_id","account_type","transaction_type","amount","city","transaction_date"])

combined_df=clean_df.unionByName(new_txn)
combined_df.orderBy("transaction_id").show()

### 16. Create a customer master DataFrame and join it with transactions
A **left** join keeps every transaction, even when the customer is missing from the master. `C110` and `C111` are left out on purpose, so their names come back null.

In [ ]:
customer_data=[
    ("C101","Arjun Rao","Retail","VERIFIED"),
    ("C102","Meera Iyer","Business","VERIFIED"),
    ("C103","Karthik Nair","Retail","PENDING"),
    ("C104","Sneha Patil","Business","VERIFIED"),
    ("C105","Rahul Verma","Retail","VERIFIED"),
    ("C106","Divya Reddy","Retail","PENDING"),
    ("C107","Vikram Shah","Retail","VERIFIED"),
    ("C108","Ananya Das","Retail","VERIFIED"),
    ("C109","Rohan Menon","Premium","VERIFIED")
]
customer_df=spark.createDataFrame(customer_data,["customer_id","customer_name","segment","kyc_status"])

joined_df=combined_df.join(customer_df,"customer_id","left")
joined_df.orderBy("transaction_id").show(truncate=False)

### 17. Repartition by account type — and why
`repartition(3, "account_type")` hash-partitions the data so **all rows of the same account type land in the same partition**.

Why this helps:
- Later `groupBy("account_type")` aggregations and window functions find each key's data already together, which cuts down later shuffles.
- Writing with `.partitionBy("account_type")` produces fewer, larger files per folder instead of many small files.
- Each account type (Savings / Current / Salary) can be processed in parallel.

Trade-off: `repartition` itself is a **full shuffle**, so use it only when the data is reused afterwards. If one account type is much bigger than the others, its partition gets skewed. To only *reduce* the partition count, use `coalesce` instead, which avoids a shuffle.

In [ ]:
print("Original partitions:",combined_df.rdd.getNumPartitions())
acct_df=combined_df.repartition(3,"account_type")
print("After repartition:",acct_df.rdd.getNumPartitions())

# which account types ended up in which partition
acct_df.withColumn("partition_id",F.spark_partition_id()) \
       .groupBy("partition_id","account_type").count() \
       .orderBy("partition_id").show()

### 18. Final banking pipeline — clean, enriched transactions
Steps: de-duplicate → remove null amounts → fill missing city → cast date → fee & net amount → risk level → rename city → enrich with customer master → select & sort.

In [ ]:
final_bank_df=(bank_df
 .unionByName(new_txn)
 .dropDuplicates(["transaction_id"])
 .filter(F.col("amount").isNotNull() & (F.col("amount")>0))
 .fillna({"city":"Unknown"})
 .withColumn("transaction_date",F.to_date("transaction_date"))
 .withColumn("transaction_fee",F.round(F.col("amount")*0.01,2))
 .withColumn("net_amount",F.round(F.col("amount")-F.col("transaction_fee"),2))
 .withColumn("risk_level",
     F.when(F.col("amount")>=100000,"HIGH")
      .when(F.col("amount")>=50000,"MEDIUM")
      .otherwise("LOW"))
 .withColumnRenamed("city","branch_city")
 .join(customer_df,"customer_id","left")
 .fillna({"customer_name":"Unknown","segment":"Unknown","kyc_status":"NOT_FOUND"})
 .select("transaction_id","customer_id","customer_name","segment","kyc_status",
         "account_type","transaction_type","amount","transaction_fee","net_amount",
         "risk_level","branch_city","transaction_date")
 .orderBy(F.col("amount").desc()))
final_bank_df.show(truncate=False)

### Challenge: Narrow vs wide transformations — where does a shuffle happen?

**Narrow**: each output partition depends on only **one** input partition, so there's no data movement across the network.
**Wide**: an output partition needs data from **many** input partitions, so Spark must **shuffle** (a new stage starts).

| Transformation used | Type | Shuffle? | Why |
|---|---|---|---|
| `select()` | Narrow | No | Picks columns row by row |
| `filter()` / `where()` | Narrow | No | Keeps or drops each row independently |
| `withColumn()` (fee, net_amount, risk_level, `to_date`, `year/month/day`) | Narrow | No | Computed from values in the same row |
| `when/otherwise` | Narrow | No | Row-level expression |
| `withColumnRenamed()` / `drop()` | Narrow | No | Only changes metadata |
| `fillna()` / `dropna()` | Narrow | No | Row-level check |
| `unionByName()` | Narrow | No | Puts the partitions of both DataFrames side by side |
| `coalesce(n)` | Narrow | No | Merges existing partitions without a full shuffle |
| `distinct()` | **Wide** | **Yes** | Identical rows must be brought together to compare them |
| `dropDuplicates(["transaction_id"])` | **Wide** | **Yes** | All rows with the same ID must meet in one partition |
| `orderBy()` / `sort()` | **Wide** | **Yes** | Global sort needs range partitioning across all data |
| `groupBy().agg()` | **Wide** | **Yes** | All rows for an account type must be co-located (Spark does a partial aggregate first to reduce shuffled data) |
| `join()` with customer master | **Wide** *(usually)* | **Yes**, unless broadcast | Both sides are shuffled by `customer_id`. Because the customer master is small, Spark will usually **broadcast** it instead (below `spark.sql.autoBroadcastJoinThreshold`, 10 MB by default), which avoids the shuffle |
| `repartition(3,"account_type")` | **Wide** | **Yes** | Explicitly redistributes every row by hash of the key |

**Shuffle points in the final pipeline (Task 18):** `dropDuplicates` → `join` (unless broadcast) → `orderBy`. Everything else runs in the same stage. You can confirm this with `final_bank_df.explain()`: look for `Exchange` operators (shuffles) and `BroadcastHashJoin` / `BroadcastExchange`.

In [ ]:
final_bank_df.explain()